---
## Important Note: Free Edition / Community Edition (serverless)

### What CAN be done (what we will do)
- `COPY INTO` from a **Unity Catalog Volume** → Delta table. Works without issue.

- Auto Loader (`cloudFiles`) with `trigger(availableNow=True)` — the only trigger supported in serverless.

- Simulate the arrival of new files **by writing CSV/JSON to the Volume** and re-running.

### What changes compared to the book
- The book uses `dbfs:/mnt/...` paths. **DBFS root and mounts are deprecated** and in the Free Edition

they are not even available. We use **Volumes**: `/Volumes/catalog/schema/volume/...`
- We don't leave a stream "running live": with Auto Loader we use `availableNow=True`, just like in streaming.

The demonstration pattern will be:
1. Generate some files in the volume (the "source").

2. Execute the ingestion (COPY INTO or Auto Loader).

3. Query the destination table.

4. Add new files and repeat → verify that it only processes new files.

The concept is the same as in the exam (incremental, idempotent, checkpoint, exactly-once); we've simply adapted the mechanism to serverless.

---
## 0. Setup

- Create catalog/schema/volume if they don't already exist.
- Define the folders within the volume:
- `landing/` → where the files to be ingested "drop" (the source).
- `checkpoints/` and `schemas/` → Auto Loader metadata.

In [0]:
# Environment variables
catalog="main"
schema = "ingestion"
volume="raw_data"

base_path = f"/Volumes/{catalog}/{schema}/{volume}"
landing_csv = f"{base_path}/landing_csv" # CSV files for COPY INTO
landing_json = f"{base_path}/landing_json" # JSON files for Auto Loader
checkpoints_path = f"{base_path}/checkpoints"
schemas_path = f"{base_path}/schemas"

print("Base path:", base_path)
print("Landing CSV:", landing_csv)
print("Landing JSON:", landing_json)
print("Checkpoints:", checkpoints_path)
print("Schemas :", schemas_path)

Base path: /Volumes/main/ingestion/raw_data
Landing CSV: /Volumes/main/ingestion/raw_data/landing_csv
Landing JSON: /Volumes/main/ingestion/raw_data/landing_json
Checkpoints: /Volumes/main/ingestion/raw_data/checkpoints
Schemas : /Volumes/main/ingestion/raw_data/schemas


In [0]:
# Create catalog/schema/volume structure if it does not exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

# Default schema to use %sql without prefix
spark.sql(f"USE {catalog}.{schema}")
print(f"List structure. Using {catalog}.{schema}")

List structure. Using main.ingestion


In [0]:
# Helper: Writing a Text File to the Volume
# In serverless environments, we can write directly to /Volumes/... using Python's open() method.
# We use this to "drop" files onto the landing page and simulate data arrival.
import os

def drop_file(folder, filename, content):
    os.makedirs(folder, exist_ok=True)
    path = f"{folder}/{filename}"
    with open(path, "w") as f:
        f.write(content)
    print(f"  + file created: {path}")

print("Helper drop_file() ready")

Helper drop_file() ready


In [0]:
# Drop the first 2 CSV files on the landing
# Format: courses with '|' delimiter and header.
csv_header = "course_id|title|instructor|category|price"

drop_file(landing_csv, "courses_001.csv", csv_header + "\n" + 
            "C01|Intro to Python|Ana R.|Programming|20\n" + 
            "C02|Data Engineering|Luis M.|Data|35\n")

drop_file(landing_csv, "courses_002.csv", csv_header + "\n" + 
            "C03|Spark Basics|Pierre B.|Data|30\n" + 
            "C04|SQL for Analytics|Julia S.|Data|25\n")

print("\n2 CSV files in the landing.")

  + file created: /Volumes/main/ingestion/raw_data/landing_csv/courses_001.csv
  + file created: /Volumes/main/ingestion/raw_data/landing_csv/courses_002.csv

2 CSV files in the landing.



## 1. What Is Incremental Ingestion?

**Data ingestion** is the process of loading data from files into Delta Lake tables.

The challenge is not loading the data itself, but doing so **without reprocessing data that has already been ingested**.

* **Traditional pipeline:** every execution rereads *all* source files → expensive, slow, and requires deduplication.
* **Incremental ingestion:** loads only the *new* files since the last run → faster and more resource-efficient.

Databricks provides **two incremental ingestion mechanisms**:

|             | `COPY INTO`                      | Auto Loader (`cloudFiles`)                   |
| ----------- | -------------------------------- | -------------------------------------------- |
| Type        | SQL statement                    | Structured Streaming source                  |
| Volume      | Thousands of files               | Millions (and beyond)                        |
| Scalability | Less efficient                   | Highly efficient                             |
| Idempotency | Yes (skips already loaded files) | Yes (checkpointing, exactly-once processing) |

The rest of this notebook focuses on understanding **when to use each approach**.



## 2. `COPY INTO` (The SQL Statement)

`COPY INTO` loads files from a source location into a Delta table in an **idempotent and incremental** manner. Each execution processes only the new files and automatically skips any files that have already been loaded.

Structure:

```sql
COPY INTO  target_table
FROM       '/path/to/files'
FILEFORMAT = CSV
FORMAT_OPTIONS (...)   -- how to parse each file (delimiter, header, ...)
COPY_OPTIONS   (...)   -- how to load the data (mergeSchema, ...)
```

First, we create the target table **empty and without defining any columns**. `COPY INTO` will populate it and **infer the schema** directly from the source files (using `inferSchema=true`).

> **Why create it without columns?** If you define fixed data types (for example, `price INT`) and the CSV schema inference detects a different type (`STRING` or `BIGINT`), `COPY INTO` fails with a `DELTA_FAILED_TO_MERGE_FIELDS` error. This happens because `mergeSchema` can **add new columns, but it cannot cast or widen an existing column type**. Allowing `COPY INTO` to infer the schema avoids this conflict.


In [0]:
%sql
-- Create an empty destination table, with NO columns
-- COPY INTO will infer the schema from the files (thanks to inferSchema=true).
CREATE TABLE IF NOT EXISTS courses_copy

In [0]:
# Execute COPY INTO (first load)
# We use f-string in spark.sql to inject the Volume path.
# Key: inferSchema='true' → infers real types (not all strings),
#      mergeSchema='true' → adapts the table schema to the inferred type.
result = spark.sql(f"""
  COPY INTO courses_copy
  FROM '{landing_csv}'
  FILEFORMAT = CSV
  FORMAT_OPTIONS ('delimiter'='|', 'header'='true', 'inferSchema'='true', 'mergeSchema'='true')
  COPY_OPTIONS   ('mergeSchema'='true')
""")

# Shows how many files/rows were uploaded
display(result)   

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


In [0]:
%sql
-- See the result: all 4 courses (C01–C04) should be there
SELECT * FROM courses_copy ORDER BY course_id

course_id,title,instructor,category,price,language
C01,Intro to Python,Ana R.,Programming,20,null
C02,Data Engineering,Luis M.,Data,35,null
C03,Spark Basics,Pierre B.,Data,30,null
C04,SQL for Analytics,Julia S.,Data,25,null
C05,Deep Learning,Bernard M.,AI,40,null
C06,MLOps,Sophie B.,AI,45,null
C07,Rust Systems,Mark H.,Programming,38,en
C08,Estadística,Ana R.,Data,22,es


### Idempotency in Action

It runs **exactly the same** `COPY INTO` command again. Since both files have already been loaded, COPY INTO **ignores them**: it doesn't duplicate anything. 
That's its guarantee of idempotency (it relies on metadata that records which files it has already processed).

In [0]:
# Re-execute COPY INTO without new files
# numAffectedRows / num_inserted_rows should be 0: do not reprocess what has already been loaded.
result = spark.sql(f"""
  COPY INTO courses_copy
  FROM '{landing_csv}'
  FILEFORMAT = CSV
  FORMAT_OPTIONS ('delimiter'='|', 'header'='true', 'inferSchema'='true', 'mergeSchema'='true')
  COPY_OPTIONS   ('mergeSchema'='true')
""")
display(result)

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


### Now a new file arrives → only process that one

We drop a third CSV file onto the landing page and re-execute COPY INTO.
Only the new file is loaded; the previous two remain ignored.

In [0]:
# Drop a new CSV file
drop_file(landing_csv, "courses_003.csv", csv_header + "\n" +
          "C05|Deep Learning|Bernard M.|AI|40\n" +
          "C06|MLOps|Sophie B.|AI|45\n")

# Re-execute COPY INTO: only loads courses_003.csv (C05, C06)
result = spark.sql(f"""
  COPY INTO courses_copy
  FROM '{landing_csv}'
  FILEFORMAT = CSV
  FORMAT_OPTIONS ('delimiter'='|', 'header'='true', 'inferSchema'='true', 'mergeSchema'='true')
  COPY_OPTIONS   ('mergeSchema'='true')
""")
display(result)

  + file created: /Volumes/main/ingestion/raw_data/landing_csv/courses_003.csv


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


In [0]:
%sql
-- Check: C01–C06 (6 courses) should now be set
SELECT * FROM courses_copy ORDER BY course_id

course_id,title,instructor,category,price,language
C01,Intro to Python,Ana R.,Programming,20,null
C02,Data Engineering,Luis M.,Data,35,null
C03,Spark Basics,Pierre B.,Data,30,null
C04,SQL for Analytics,Julia S.,Data,25,null
C05,Deep Learning,Bernard M.,AI,40,null
C06,MLOps,Sophie B.,AI,45,null
C07,Rust Systems,Mark H.,Programming,38,en
C08,Estadística,Ana R.,Data,22,es


---
## 3. `mergeSchema` (schema evolution)

`COPY_OPTIONS('mergeSchema'='true')` allows the table **to adapt** if the structure 
of the incoming data changes (for example, a new column), instead of breaking.

We drop in a file with an **extra** `language` **column** and let the table evolve.

In [0]:
# File with a new column: 'language'
csv_header_v2 = "course_id|title|instructor|category|price|language"

drop_file(landing_csv, "courses_004.csv", csv_header_v2 + "\n" +
          "C07|Rust Systems|Mark H.|Programming|38|en\n" +
          "C08|Estadística|Ana R.|Data|22|es\n")

# With mergeSchema=true, the table automatically adds the 'language' column.
result = spark.sql(f"""
  COPY INTO courses_copy
  FROM '{landing_csv}'
  FILEFORMAT = CSV
  FORMAT_OPTIONS ('delimiter'='|', 'header'='true', 'inferSchema'='true', 'mergeSchema'='true')
  COPY_OPTIONS   ('mergeSchema'='true')
""")
display(result)

  + file created: /Volumes/main/ingestion/raw_data/landing_csv/courses_004.csv


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


In [0]:
%sql
-- The table now has the 'language' column (null in the old rows)
SELECT * FROM courses_copy ORDER BY course_id

course_id,title,instructor,category,price,language
C01,Intro to Python,Ana R.,Programming,20,null
C02,Data Engineering,Luis M.,Data,35,null
C03,Spark Basics,Pierre B.,Data,30,null
C04,SQL for Analytics,Julia S.,Data,25,null
C05,Deep Learning,Bernard M.,AI,40,null
C06,MLOps,Sophie B.,AI,45,null
C07,Rust Systems,Mark H.,Programming,38,en
C08,Estadística,Ana R.,Data,22,es


> **Exam Point:** `mergeSchema` prevents the pipeline from breaking when the incoming schema changes. 
> Without it, a file with different columns would fail.
> `COPY INTO` is ideal when the volume is manageable (thousands of files) and you want simplicity in SQL.

---
## 4. Auto Loader (the `cloudFiles` Format)

The second, more powerful ingestion method is built **on top of Structured Streaming**. As a result, it inherits all streaming capabilities, including micro-batch processing, checkpointing, and fault tolerance.

* **Massive scalability:** capable of handling billions of files and millions of new files per hour.
* **Checkpointing:** keeps track of which files have already been processed, providing **exactly-once** guarantees.
* **Fault tolerance:** if the stream fails, it resumes from the exact point where it stopped.

The special reader used by Auto Loader is the **`cloudFiles`** format, which relies on three key options:

| Option                        | Purpose                                                              |
| ----------------------------- | -------------------------------------------------------------------- |
| `cloudFiles.format`           | Specifies the source file format (`json`, `csv`, `parquet`, etc.)    |
| `cloudFiles.inferColumnTypes` | Infers column data types automatically                               |
| `cloudFiles.schemaLocation`   | Stores the inferred schema to avoid re-inferring it on every startup |

In this example, we will ingest **JSON files** using Auto Loader. First, we drop 2 files into the source location.

In [0]:
# Drop 2 JSON files into the Auto Loader landing page
# Line-delimited JSON (one object per line).
drop_file(landing_json, "events_001.json",
          '{"event_id":"E01","user":"ana","action":"login","ts":1700000001}\n'
          '{"event_id":"E02","user":"luis","action":"click","ts":1700000002}\n')

drop_file(landing_json, "events_002.json",
          '{"event_id":"E03","user":"pierre","action":"logout","ts":1700000003}\n'
          '{"event_id":"E04","user":"julia","action":"login","ts":1700000004}\n')

print("\n2 JSON files on the landing.")

  + file created: /Volumes/main/ingestion/raw_data/landing_json/events_001.json
  + file created: /Volumes/main/ingestion/raw_data/landing_json/events_002.json

2 JSON files on the landing.


### Schema in Auto Loader (Exam Point)

- For formats **with defined types** (Parquet), Auto Loader extracts the schema directly from the file.

- For formats **without types** (JSON, CSV), by default it infers **all columns as `string`**, unless you enable `cloudFiles.inferColumnTypes`.

Here we enable `inferColumnTypes` so that `ts` is displayed as a number and not as a string.

In [0]:
# Define the Auto Loader stream
# .readStream + format("cloudFiles"). It's not reading yet: define the plan.
ckpt_events   = f"{checkpoints_path}/events"
schema_events = f"{schemas_path}/events"

events_stream_df = (
    spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .option("cloudFiles.inferColumnTypes", "true")
      # save the inferred schema
      .option("cloudFiles.schemaLocation", schema_events)   
      .load(landing_json)
)

print("Is it streaming?", events_stream_df.isStreaming)   # True
events_stream_df.printSchema()

Is it streaming? True
root
 |-- action: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- ts: long (nullable = true)
 |-- user: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
# Persist with writeStream + availableNow (single trigger in serverless)
# availableNow processes everything available in one or more micro-batches and stops.
(
  events_stream_df.writeStream
    .trigger(availableNow=True)
    # checkpoint = which files it has already processed
    .option("checkpointLocation", ckpt_events)   
    .toTable("events_auto")
    # wait for the batch to finish
    .awaitTermination()                          
)
print("Auto Loader: first batch completed")

Auto Loader: first batch completed


In [0]:
%sql
-- See the result: 4 events (E01–E04)
SELECT * FROM events_auto ORDER BY event_id

action,event_id,ts,user,_rescued_data
login,E01,1700000001,ana,null
click,E02,1700000002,luis,null
logout,E03,1700000003,pierre,null
login,E04,1700000004,julia,null
click,E05,1700000005,sophie,null
login,E06,1700000006,mark,null


### Incremental Auto Loader: A new file arrives

We drop a third JSON file and run the same `writeStream` again with `availableNow=True`. Thanks to the checkpoint, Auto Loader knows that `events_001` and `events_002` have already been processed and only ingests the new file exactly once.

In [0]:
# Drop a new JSON
drop_file(landing_json, "events_003.json",
          '{"event_id":"E05","user":"sophie","action":"click","ts":1700000005}\n'
          '{"event_id":"E06","user":"mark","action":"login","ts":1700000006}\n')

# Re-launch the stream: the checkpoint causes it to only process events_003.json
(
  events_stream_df.writeStream
    .trigger(availableNow=True)
    .option("checkpointLocation", ckpt_events) 
    .toTable("events_auto")
    .awaitTermination()
)
print("Auto Loader: Incremental batch completed (new file only)")

  + file created: /Volumes/main/ingestion/raw_data/landing_json/events_003.json
Auto Loader: Incremental batch completed (new file only)


In [0]:
%sql
-- Check: now 6 events (E01–E06)
SELECT * FROM events_auto ORDER BY event_id

action,event_id,ts,user,_rescued_data
login,E01,1700000001,ana,null
click,E02,1700000002,luis,null
logout,E03,1700000003,pierre,null
login,E04,1700000004,julia,null
click,E05,1700000005,sophie,null
login,E06,1700000006,mark,null


> **The key distinction in the code** is the use of the `cloudFiles` format together with `schemaLocation`.
>
> Notice that the execution pattern (`writeStream` + `availableNow` + `checkpoint`). Auto Loader *is* Structured Streaming—the only difference is that the source is a directory of files rather than a Delta table.

---

## 5. Comparison (Which One Should You Use?)

The decision comes down to two factors: **file volume** and **scalability efficiency**.

| Criterion        | `COPY INTO`              | Auto Loader                       |
| ---------------- | ------------------------ | --------------------------------- |
| Volume           | Thousands of files       | Millions of files                 |
| Scalability      | Less efficient           | Highly efficient (micro-batches)  |
| Interface        | Pure SQL                 | Structured Streaming (Python/SQL) |
| State Management | Metadata of loaded files | Checkpoint + schemaLocation       |
| Guarantee        | Idempotent               | Exactly-once                      |

**Rule of thumb:**

* **Small, fixed workloads → `COPY INTO`**
* **Growing, continuous, or large-scale workloads → Auto Loader**

Databricks' **official recommendation** is to use Auto Loader when ingesting data from cloud object storage. In fact, the `COPY INTO` documentation now recommends Auto Loader when the source directory contains a very large number of files.

> **2026 Note:** For production workloads, **file notification mode** (using file events) is preferred over directory listing. The `cloudFiles.useIncrementalListing` option has been deprecated (default is now `false`). In Databricks Free Edition, we use directory listing because it is sufficient for learning the core concepts.


---
## Clean Up

In [0]:
def clean_up():
    # Stop any active streams that may still be running
    for s in spark.streams.active:
        print(f"Stopping stream: {s.name}")
        s.stop()

    print("Dropping tables...")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.courses_copy")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.events_auto")

    print("Removing files from the volume...")
    dbutils.fs.rm(landing_csv,      True)
    dbutils.fs.rm(landing_json,     True)
    dbutils.fs.rm(checkpoints_path, True)
    dbutils.fs.rm(schemas_path,     True)

    print("Dropping schema...")
    spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema} CASCADE")

    print("Done ✓")


In [0]:
# clean_up()